In [6]:
import requests
import time
import pandas as pd

url = "https://api.openalex.org/works"
queries = [
    "Litopenaeus vannamei juvenile dietary protein requirement",
    "Fenneropenaeus chinensis protein requirement juvenile shrimp",
    "Marsupenaeus japonicus kuruma shrimp protein requirement",
    "Penaeus monodon juvenile protein requirement",
    "Homarus americanus lobster juvenile protein requirement diet",
    "Panulirus spiny lobster juvenile protein requirement diet",
]

all_papers = []
for q in queries:
    resp = requests.get(
        url,
        params={"search": q, "per_page": 25},  # filter 없이, 넉넉히 25개 받기
        headers={"User-Agent": "mailto:test@example.com"},
    )
    print(q, resp.status_code)
    if resp.status_code == 200:
        data = resp.json()
        for w in data.get("results", []):
            journal = None
            if w.get("primary_location") and w["primary_location"].get("source"):
                journal = w["primary_location"]["source"].get("display_name")
            all_papers.append({
                "query": q,
                "title": w.get("title"),
                "year": w.get("publication_year"),
                "citations": w.get("cited_by_count"),
                "journal": journal,
                "doi": w.get("doi"),
            })
    time.sleep(1.5)

papers_df = pd.DataFrame(all_papers)
print("전체 수집:", len(papers_df))

# 저널명에 "Aquacult"(Aquaculture, Aquaculture

Litopenaeus vannamei juvenile dietary protein requirement 200
Fenneropenaeus chinensis protein requirement juvenile shrimp 200
Marsupenaeus japonicus kuruma shrimp protein requirement 200
Penaeus monodon juvenile protein requirement 200
Homarus americanus lobster juvenile protein requirement diet 200
Panulirus spiny lobster juvenile protein requirement diet 200
전체 수집: 150


In [7]:
papers_df["journal"] = papers_df["journal"].fillna("")
aqua_df = papers_df[papers_df["journal"].str.contains("Aquacult", case=False)]
print("Aquaculture 계열 저널만:", len(aqua_df))

aqua_df.to_csv("shrimp_protein_papers_v3.csv", index=False, encoding="utf-8-sig")
aqua_df[["query", "title", "year", "citations", "journal"]]

Aquaculture 계열 저널만: 53


,query,title,year,citations,journal
0,Litopenaeus vannamei juvenile dietary protein ...,Effects of bioflocs on dietary protein require...,2015,46,Aquaculture Research
1,Litopenaeus vannamei juvenile dietary protein ...,Dietary lysine requirement of juvenile Pacific...,2012,119,Aquaculture
2,Litopenaeus vannamei juvenile dietary protein ...,Dietary arginine requirement of juvenile Pacif...,2012,96,Aquaculture
3,Litopenaeus vannamei juvenile dietary protein ...,Dietary threonine requirements of juvenile Pac...,2013,88,Aquaculture
4,Litopenaeus vannamei juvenile dietary protein ...,Evaluation of black soldier fly (Hermetia illu...,2017,257,Aquaculture
5,Litopenaeus vannamei juvenile dietary protein ...,Protein requirement for maintenance and maximu...,2002,175,Aquaculture
9,Litopenaeus vannamei juvenile dietary protein ...,Substitution of fish meal with plant protein s...,2009,177,Aquaculture
10,Litopenaeus vannamei juvenile dietary protein ...,Dietary magnesium requirement and physiologica...,2005,64,Aquaculture Nutrition
11,Litopenaeus vannamei juvenile dietary protein ...,Effect of dietary protein and energy levels on...,2001,83,Aquaculture Research
12,Litopenaeus vannamei juvenile dietary protein ...,Quantitative dietary threonine requirement of ...,2009,41,Aquaculture Research


In [8]:
import re

def get_abstract(work_id):
    """OpenAlex는 초록을 inverted index 형태로 줘서, 원문으로 복원해야 함"""
    resp = requests.get(work_id, headers={"User-Agent": "mailto:test@example.com"})
    if resp.status_code != 200:
        return None
    data = resp.json()
    inv_index = data.get("abstract_inverted_index")
    if not inv_index:
        return None
    # inverted index -> 원문 복원
    max_pos = max(pos for positions in inv_index.values() for pos in positions)
    words = [""] * (max_pos + 1)
    for word, positions in inv_index.items():
        for pos in positions:
            words[pos] = word
    return " ".join(words)

def extract_protein_pct(abstract):
    if not abstract:
        return []
    # "protein" 앞뒤 60자 이내에 있는 숫자(%)를 찾음
    matches = []
    for m in re.finditer(r"(\d{1,2}(?:\.\d+)?)\s*%", abstract):
        start = max(0, m.start() - 60)
        end = min(len(abstract), m.end() + 60)
        context = abstract[start:end]
        if "protein" in context.lower():
            matches.append((m.group(1), context.strip()))
    return matches

# aqua_df의 각 논문에 대해 초록 가져와서 추출 (시간이 걸리니 상위 몇 편만 테스트)
results = []
for idx, row in aqua_df.head(15).iterrows():  # 먼저 15편만 테스트
    work_url = f"https://api.openalex.org/works/{row['doi']}" if row['doi'] else None
    if not work_url:
        continue
    abstract = get_abstract(work_url)
    pct_matches = extract_protein_pct(abstract)
    results.append({
        "title": row["title"],
        "year": row["year"],
        "journal": row["journal"],
        "protein_pct_candidates": pct_matches
    })
    time.sleep(1)

for r in results:
    print(f"\n[{r['year']}] {r['title'][:70]}")
    if r["protein_pct_candidates"]:
        for pct, ctx in r["protein_pct_candidates"]:
            print(f"  -> {pct}% 근처 문맥: ...{ctx}...")
    else:
        print("  -> 초록에서 단백질 % 언급을 찾지 못함")


[2015] Effects of bioflocs on dietary protein requirement in juvenile whitele
  -> 25% 근처 문맥: ...nts (BFT) and one control group were managed: BFT fed diets 25% of crude protein (CP) (BFT-25%), 30% CP (BFT-30%), 35% CP (...
  -> 25% 근처 문맥: ...were managed: BFT fed diets 25% of crude protein (CP) (BFT-25%), 30% CP (BFT-30%), 35% CP (BFT-35%) and 40% CP (BFT-40%),...
  -> 30% 근처 문맥: ...managed: BFT fed diets 25% of crude protein (CP) (BFT-25%), 30% CP (BFT-30%), 35% CP (BFT-35%) and 40% CP (BFT-40%), and cl...
  -> 30% 근처 문맥: ...fed diets 25% of crude protein (CP) (BFT-25%), 30% CP (BFT-30%), 35% CP (BFT-35%) and 40% CP (BFT-40%), and clear water co...
  -> 35% 근처 문맥: ...iets 25% of crude protein (CP) (BFT-25%), 30% CP (BFT-30%), 35% CP (BFT-35%) and 40% CP (BFT-40%), and clear water control...
  -> 35% 근처 문맥: ...crude protein (CP) (BFT-25%), 30% CP (BFT-30%), 35% CP (BFT-35%) and 40% CP (BFT-40%), and clear water control without biof...
  -> 40% 근처 문맥: ...ol. Total protein level in the

In [9]:
q_tor = "TOR signaling pathway dietary protein requirement shrimp growth"

resp = requests.get(
    url,
    params={"search": q_tor, "per_page": 15},
    headers={"User-Agent": "mailto:test@example.com"},
)
print(resp.status_code)

tor_papers = []
if resp.status_code == 200:
    for w in resp.json().get("results", []):
        journal = None
        if w.get("primary_location") and w["primary_location"].get("source"):
            journal = w["primary_location"]["source"].get("display_name")
        tor_papers.append({
            "title": w.get("title"),
            "year": w.get("publication_year"),
            "citations": w.get("cited_by_count"),
            "journal": journal,
            "doi": w.get("doi"),
        })

tor_df = pd.DataFrame(tor_papers)
tor_df

200


,title,year,citations,journal,doi
0,Leucine promotes protein synthesis of juvenile...,2022,30,Aquaculture,https://doi.org/10.1016/j.aquaculture.2022.739060
1,"Growth Performance, Digestive Enzymes, and TOR...",2018,33,Frontiers in Physiology,https://doi.org/10.3389/fphys.2018.00998
2,"Effect of dietary arginine on growth, intestin...",2011,172,British Journal Of Nutrition,https://doi.org/10.1017/s0007114511005459
3,Dietary Lysine Levels Improved Antioxidant Cap...,2021,57,Frontiers in Immunology,https://doi.org/10.3389/fimmu.2021.635015
4,"Growth, physiological, biochemical, and molecu...",2021,61,Aquaculture,https://doi.org/10.1016/j.aquaculture.2021.736393
5,"Dietary isoleucine improved flesh quality, mus...",2021,63,Journal of Animal Science and Biotechnology/Jo...,https://doi.org/10.1186/s40104-021-00572-4
6,"Dietary valine improved growth, immunity, enzy...",2021,39,Scientific Reports,https://doi.org/10.1038/s41598-021-01142-4
7,Effects of Replacement of Dietary Fishmeal by ...,2021,73,Frontiers in Physiology,https://doi.org/10.3389/fphys.2021.764987
8,Dietary β-Hydroxy-β-Methylbutyrate Supplementa...,2023,22,Aquaculture Nutrition,https://doi.org/10.1155/2023/9889533
9,Dietary Lysine Regulates Body Growth Performan...,2020,37,Frontiers in Marine Science,https://doi.org/10.3389/fmars.2020.595682


In [10]:
q_digest = [
    "trypsin chymotrypsin digestive enzyme shrimp crustacean protein digestion",
    "digestive protease activity growth performance shrimp dietary protein utilization",
    "trypsin activity crustacean digestive gland protein hydrolysis",
]

digest_papers = []
for q in q_digest:
    resp = requests.get(
        url,
        params={"search": q, "per_page": 15},
        headers={"User-Agent": "mailto:test@example.com"},
    )
    print(q, resp.status_code)
    if resp.status_code == 200:
        for w in resp.json().get("results", []):
            journal = None
            if w.get("primary_location") and w["primary_location"].get("source"):
                journal = w["primary_location"]["source"].get("display_name")
            digest_papers.append({
                "query": q,
                "title": w.get("title"),
                "year": w.get("publication_year"),
                "citations": w.get("cited_by_count"),
                "journal": journal,
                "doi": w.get("doi"),
            })
    time.sleep(1.5)

digest_df = pd.DataFrame(digest_papers)
print("전체:", len(digest_df))

# 저널 필터 (Aquaculture, Comparative Biochemistry, Fish Physiology 등 관련 저널 위주로)
digest_df["journal"] = digest_df["journal"].fillna("")
relevant_journals = "Aquacult|Comparative Biochemistry|Fish Physiology|Marine Biology|Crustacean"
filtered_df = digest_df[digest_df["journal"].str.contains(relevant_journals, case=False, regex=True)]
print("관련 저널만:", len(filtered_df))

filtered_df.to_csv("digestive_enzyme_papers.csv", index=False, encoding="utf-8-sig")
filtered_df[["query", "title", "year", "citations", "journal"]]

trypsin chymotrypsin digestive enzyme shrimp crustacean protein digestion 200
digestive protease activity growth performance shrimp dietary protein utilization 200
trypsin activity crustacean digestive gland protein hydrolysis 200
전체: 45
관련 저널만: 18


,query,title,year,citations,journal
0,trypsin chymotrypsin digestive enzyme shrimp c...,Digestive enzymes present in crustacean feces ...,2003,63,Journal of Experimental Marine Biology and Eco...
1,trypsin chymotrypsin digestive enzyme shrimp c...,Feeding behaviour and digestive physiology in ...,2013,495,Reviews in Aquaculture
2,trypsin chymotrypsin digestive enzyme shrimp c...,Comparison of digestive proteinases in three p...,2011,29,Aquaculture
9,trypsin chymotrypsin digestive enzyme shrimp c...,Trypsin Synthesis and Storage as Zymogen in th...,2004,40,Journal of Crustacean Biology
11,trypsin chymotrypsin digestive enzyme shrimp c...,Digestive enzymes in the ontogenetic stages of...,2006,43,Marine Biology
13,trypsin chymotrypsin digestive enzyme shrimp c...,Digestive Enzyme Activities in the Alimentary ...,2001,50,Journal of Crustacean Biology
16,digestive protease activity growth performance...,Effect of dietary components on the gut microb...,2015,709,Aquaculture Nutrition
17,digestive protease activity growth performance...,Feeding behaviour and digestive physiology in ...,2013,495,Reviews in Aquaculture
19,digestive protease activity growth performance...,"Effects of crystalline amino acids, phytase an...",2015,52,Aquaculture
20,digestive protease activity growth performance...,Probiotics and competitive exclusion of pathog...,2020,184,Reviews in Aquaculture


In [13]:
import re

def get_abstract(doi):
    """DOI로 OpenAlex work 조회 후 초록 원문 복원"""
    if not doi:
        return None
    work_url = f"https://api.openalex.org/works/{doi}"
    resp = requests.get(work_url, headers={"User-Agent": "mailto:test@example.com"})
    if resp.status_code != 200:
        return None
    data = resp.json()
    inv_index = data.get("abstract_inverted_index")
    if not inv_index:
        return None
    max_pos = max(pos for positions in inv_index.values() for pos in positions)
    words = [""] * (max_pos + 1)
    for word, positions in inv_index.items():
        for pos in positions:
            words[pos] = word
    return " ".join(words)

# 문장이 담고 있는 핵심 키워드들 (가중치를 다르게 줄 수도 있음)
key_terms = {
    "trypsin": 2,
    "chymotrypsin": 2,
    "digestive enzyme": 2,
    "protease": 1,
    "protein digestion": 2,
    "protein utilization": 2,
    "growth": 1,
    "shrimp": 1,
    "crustacean": 1,
}

def score_abstract(abstract):
    if not abstract:
        return 0, []
    abstract_lower = abstract.lower()
    matched = []
    score = 0
    for term, weight in key_terms.items():
        if term in abstract_lower:
            score += weight
            matched.append(term)
    return score, matched

# filtered_df (이전 단계 결과)의 각 논문에 대해 초록 받아서 점수 매기기
results = []
for idx, row in filtered_df.iterrows():
    abstract = get_abstract(row["doi"])
    score, matched = score_abstract(abstract)
    results.append({
        "title": row["title"],
        "year": row["year"],
        "journal": row["journal"],
        "doi": row["doi"],
        "relevance_score": score,
        "matched_terms": ", ".join(matched),
        "abstract_snippet": (abstract[:200] + "...") if abstract else "(초록 없음)"
    })
    time.sleep(1)

result_df = pd.DataFrame(results).sort_values("relevance_score", ascending=False)
result_df.to_csv("digestive_enzyme_scored.csv", index=False, encoding="utf-8-sig")
result_df[["title", "year", "journal", "relevance_score", "matched_terms"]]

,title,year,journal,relevance_score,matched_terms
13,RELATIONSHIP BETWEEN DIETARY PREFERENCES AND D...,1998,Journal of Crustacean Biology,8,"trypsin, chymotrypsin, digestive enzyme, prote..."
5,Digestive Enzyme Activities in the Alimentary ...,2001,Journal of Crustacean Biology,7,"trypsin, chymotrypsin, digestive enzyme, protease"
17,Digestive Enzyme Activities in the Alimentary ...,2001,Journal of Crustacean Biology,7,"trypsin, chymotrypsin, digestive enzyme, protease"
3,Trypsin Synthesis and Storage as Zymogen in th...,2004,Journal of Crustacean Biology,3,"trypsin, shrimp"
16,Trypsin Synthesis and Storage as Zymogen in th...,2004,Journal of Crustacean Biology,3,"trypsin, shrimp"
9,Probiotics and competitive exclusion of pathog...,2020,Reviews in Aquaculture,2,"growth, shrimp"
1,Feeding behaviour and digestive physiology in ...,2013,Reviews in Aquaculture,0,
0,Digestive enzymes present in crustacean feces ...,2003,Journal of Experimental Marine Biology and Eco...,0,
4,Digestive enzymes in the ontogenetic stages of...,2006,Marine Biology,0,
2,Comparison of digestive proteinases in three p...,2011,Aquaculture,0,


In [15]:
result_df_unique = result_df.drop_duplicates(subset="title")
result_df_unique[["title", "year", "journal", "relevance_score", "matched_terms"]]

,title,year,journal,relevance_score,matched_terms
13,RELATIONSHIP BETWEEN DIETARY PREFERENCES AND D...,1998,Journal of Crustacean Biology,8,"trypsin, chymotrypsin, digestive enzyme, prote..."
5,Digestive Enzyme Activities in the Alimentary ...,2001,Journal of Crustacean Biology,7,"trypsin, chymotrypsin, digestive enzyme, protease"
3,Trypsin Synthesis and Storage as Zymogen in th...,2004,Journal of Crustacean Biology,3,"trypsin, shrimp"
9,Probiotics and competitive exclusion of pathog...,2020,Reviews in Aquaculture,2,"growth, shrimp"
1,Feeding behaviour and digestive physiology in ...,2013,Reviews in Aquaculture,0,
0,Digestive enzymes present in crustacean feces ...,2003,Journal of Experimental Marine Biology and Eco...,0,
4,Digestive enzymes in the ontogenetic stages of...,2006,Marine Biology,0,
2,Comparison of digestive proteinases in three p...,2011,Aquaculture,0,
8,"Effects of crystalline amino acids, phytase an...",2015,Aquaculture,0,
6,Effect of dietary components on the gut microb...,2015,Aquaculture Nutrition,0,


In [16]:
result_df_unique[["title", "doi"]].head(3)

,title,doi
13,RELATIONSHIP BETWEEN DIETARY PREFERENCES AND D...,https://doi.org/10.1163/193724098x00511
5,Digestive Enzyme Activities in the Alimentary ...,https://doi.org/10.1163/20021975-99990133
3,Trypsin Synthesis and Storage as Zymogen in th...,https://doi.org/10.1651/c-2423
